# Credit Score Prediction – Project Submission

**Prepared by:** Himanshu Shrivastava  
**Project:** Exploratory Data Analysis and Credit Score Classification  
**Dataset:** `dataset-2.csv`  
**Target column:** `Credit_Score` (`Poor`, `Standard`, `Good`)

## Project note

In this project, I analyzed customer financial and payment-related information and built a model to predict one of three credit-score categories. I used Python for data checking, charts, preprocessing, model training, and evaluation.

The supplied brief has the title **“Paisabazaar Banking Fraud Analysis”**, but its business-context section describes creditworthiness and predicting credit-score categories. The supplied dataset also contains `Credit_Score`, not a fraud label. I have therefore focused on **credit-score prediction** and have not described this work as fraud detection.

## 1. My objective and approach

My main objective was to understand the dataset and test whether customer financial information can help predict the recorded credit-score category.

I followed these steps:

1. Inspect the dataset, its columns, data types, and data quality.
2. Explore the target classes and financial variables with different charts.
3. Handle known placeholder values and prepare the input features.
4. Split the data by customer so the same customer is not in both the training and test sets.
5. Compare a simple baseline with Random Forest and Logistic Regression.
6. Review the model results, check the effect of `Credit_Mix`, and save the final model and reports.

I treated this as a **multiclass classification** problem because the target contains categories, not a continuous number. The results in this notebook are for this dataset and split only; they are not proof that the model is ready to make real lending decisions.

## 2. Import the libraries

I used pandas to work with the data, Matplotlib and Seaborn to create charts, and scikit-learn to train and evaluate the models. I fixed the random seed where applicable so that the main results can be reproduced.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
CLASS_ORDER = ["Poor", "Standard", "Good"]

pd.set_option("display.max_columns", 50)

## 3. Load the dataset from Google Drive

I saved `dataset-2.csv` in the `Credit_Score_Project` folder in my Google Drive. This cell mounts Drive and reads that file. I keep the dataset outside the notebook so the notebook stays easy to update if the CSV changes.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

file_path = "/content/drive/MyDrive/Credit_Score_Project/dataset-2.csv"
df = pd.read_csv(file_path)

print("Dataset loaded successfully")
print("File path:", file_path)
print("Dataset shape:", df.shape)

## 4. Inspect the data

I checked the first and last few records, the column data types, and summary statistics before making changes. This helped me confirm the dataset structure and spot any data-quality questions.

In [ ]:
print("First five rows:")
display(df.head())

print("Last five rows:")
display(df.tail())

print("Column names and data types:")
df.info()

print("Numerical summary:")
display(df.describe().T.round(2))

## 5. Data dictionary

The file did not come with a separate formal data dictionary. I wrote the descriptions below from the column names and observed values, so the exact business meaning and units of some fields should be confirmed with the data provider.

| Column | My interpretation and use in this project |
|---|---|
| `ID` | Row identifier; not used as a predictor. |
| `Customer_ID` | Customer identifier; used to keep each customer's records together during the split, not used as a predictor. |
| `Month` | Month index in the supplied records; not used as a predictor in this model. |
| `Name` | Customer name field; excluded as an identifier. |
| `Age` | Customer age value; used as a numerical feature after a basic data-quality check. |
| `SSN` | Personal identifier; excluded from the model. |
| `Occupation` | Occupation category; used as a categorical feature. |
| `Annual_Income` | Annual income value; used as a numerical feature. |
| `Monthly_Inhand_Salary` | Monthly in-hand salary value; used as a numerical feature. |
| `Num_Bank_Accounts` | Number of bank accounts; numerical feature. |
| `Num_Credit_Card` | Number of credit cards; numerical feature. |
| `Interest_Rate` | Recorded interest-rate value; numerical feature. |
| `Num_of_Loan` | Number of loans; numerical feature. |
| `Type_of_Loan` | Combined loan-type text; excluded because it has many distinct combinations and would require separate text/feature engineering. |
| `Delay_from_due_date` | Recorded delay from due date; numerical feature. |
| `Num_of_Delayed_Payment` | Number of delayed payments; numerical feature. |
| `Changed_Credit_Limit` | Recorded credit-limit change; numerical feature. |
| `Num_Credit_Inquiries` | Number of credit inquiries; numerical feature. |
| `Credit_Mix` | Credit-mix category; categorical feature, also checked separately for sensitivity. |
| `Outstanding_Debt` | Outstanding debt value; numerical feature. |
| `Credit_Utilization_Ratio` | Credit-utilization value; numerical feature. |
| `Credit_History_Age` | Credit-history age value as stored; numerical feature. The source file does not define its unit. |
| `Payment_of_Min_Amount` | Minimum-payment status; categorical feature. |
| `Total_EMI_per_month` | Monthly EMI value; numerical feature. |
| `Amount_invested_monthly` | Monthly investment value; numerical feature. |
| `Payment_Behaviour` | Payment/spending behaviour category; categorical feature. |
| `Monthly_Balance` | Monthly balance value; numerical feature. |
| `Credit_Score` | Target variable to predict: `Poor`, `Standard`, or `Good`. |

## 6. Check data quality and the target classes

I checked for actual null values, exact duplicate rows, and the distribution of the target. A column can contain text placeholders even when pandas reports no null values, so I also handle known placeholders in the next section.

In [ ]:
print("Missing values in the original file:")
display(df.isna().sum().sort_values(ascending=False).head(15))

print("Number of exact duplicate rows:", df.duplicated().sum())

print("Credit-score counts:")
display(df["Credit_Score"].value_counts())

print("Credit-score percentages:")
display((df["Credit_Score"].value_counts(normalize=True) * 100).round(2))

print("Unique customers:", df["Customer_ID"].nunique())
print("Records per customer:")
display(df.groupby("Customer_ID").size().value_counts().sort_index())

## 7. Clean known placeholder values

I made a separate cleaned copy so the original data remains unchanged. I changed the exact placeholders `NM`, `No Data`, `Not Specified`, and `_______` to missing values. I also marked age values below 18 as missing for this adult credit-applicant analysis. That age rule is an assumption because the supplied file does not explain these records; I have kept it visible rather than silently treating those values as valid.

I did not remove duplicate rows because the original file has no exact duplicate rows. Missing numerical and categorical values are handled inside the model pipelines using information learned from the training data.

In [ ]:
clean_df = df.copy()

placeholder_values = ["NM", "No Data", "Not Specified", "_______"]
for column in clean_df.select_dtypes(include="object").columns:
    clean_df[column] = clean_df[column].replace(placeholder_values, np.nan)

under_18_count = (clean_df["Age"] < 18).sum()
clean_df.loc[clean_df["Age"] < 18, "Age"] = np.nan

print("Age values below 18 marked as missing:", under_18_count)
print("Missing values after cleaning:")
display(clean_df.isna().sum().sort_values(ascending=False).head(15))

## 8. Exploratory data analysis

I used different chart types to examine the target distribution, numerical variables, and relationships between features and credit-score categories. These charts show associations in this dataset; they do not prove that one feature causes a credit-score outcome.

### Chart 1. Distribution of credit-score classes

I used a bar chart to see whether the three classes are equally represented. This matters because accuracy alone can hide weak performance on a less frequent class.

In [ ]:
plt.figure(figsize=(7, 4))
score_counts = clean_df["Credit_Score"].value_counts().reindex(CLASS_ORDER)
plt.bar(score_counts.index, score_counts.values)
plt.title("Distribution of Credit Score Classes")
plt.xlabel("Credit score")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

### Chart 2. Annual income distribution

I used a histogram to see the spread of annual income and whether the values are concentrated in a smaller range or extend toward higher amounts.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(clean_df["Annual_Income"].dropna(), bins=30, color="steelblue")
plt.title("Annual Income Distribution")
plt.xlabel("Annual income")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

### Chart 3. Age distribution after the data-quality check

I used a histogram to see the age values remaining after treating ages below 18 as missing. I kept this rule explicit because it is an assumption about the intended applicant population.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(clean_df["Age"].dropna(), bins=25, color="steelblue")
plt.title("Age Distribution After Basic Cleaning")
plt.xlabel("Age")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

### Chart 4. Payment delay by credit-score class

I used boxplots to compare the middle value and spread of `Delay_from_due_date` across the three credit-score classes. I used a boxplot because it makes group differences and extreme values easier to inspect.

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=clean_df, x="Credit_Score", y="Delay_from_due_date", order=CLASS_ORDER)
plt.title("Payment Delay by Credit Score")
plt.xlabel("Credit score")
plt.ylabel("Delay from due date")
plt.tight_layout()
plt.show()

### Chart 5. Credit Mix and credit-score class

I used a 100% stacked bar chart to compare the credit-score proportions within each `Credit_Mix` category. I used percentages so that categories with different record counts could be compared more fairly.

In [ ]:
credit_mix_share = pd.crosstab(
    clean_df["Credit_Mix"],
    clean_df["Credit_Score"],
    normalize="index"
) * 100
credit_mix_share = credit_mix_share.reindex(columns=CLASS_ORDER)

ax = credit_mix_share.plot(kind="bar", stacked=True, figsize=(8, 4))
ax.set_title("Credit Score Mix Within Each Credit_Mix Category")
ax.set_xlabel("Credit Mix")
ax.set_ylabel("Share within category (%)")
ax.legend(title="Credit score", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Chart 6. Correlation heatmap for numerical features

I used a correlation heatmap to see which numerical fields move together in a linear way. I removed identifier fields from this chart. Correlation does not show causation, and it does not fully describe relationships between categorical variables.

In [ ]:
numeric_data = clean_df.select_dtypes(include="number").drop(
    columns=["ID", "Customer_ID", "SSN", "Month"],
    errors="ignore"
)

plt.figure(figsize=(12, 8))
sns.heatmap(numeric_data.corr(), cmap="coolwarm", center=0)
plt.title("Correlation Between Numerical Variables")
plt.tight_layout()
plt.show()

### Chart 7. Annual income and outstanding debt

I used a scatterplot to look at annual income and outstanding debt together. I plotted a random sample of up to 3,000 records to keep the chart readable. The colour shows the recorded credit-score class.

In [ ]:
scatter_sample = clean_df.sample(n=min(3000, len(clean_df)), random_state=RANDOM_STATE)

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=scatter_sample,
    x="Annual_Income",
    y="Outstanding_Debt",
    hue="Credit_Score",
    hue_order=CLASS_ORDER,
    alpha=0.6,
    s=25
)
plt.title("Annual Income vs Outstanding Debt")
plt.xlabel("Annual income")
plt.ylabel("Outstanding debt")
plt.tight_layout()
plt.show()

## 9. What I learned from the charts

The next tables show the main patterns numerically so I do not rely on visual impressions alone. I use the word “associated” rather than claiming that one variable causes a credit-score category.

In [ ]:
print("Credit-score distribution (%):")
score_percentages = (clean_df["Credit_Score"].value_counts(normalize=True) * 100).reindex(CLASS_ORDER)
display(score_percentages.round(2).rename("Percentage"))

print("Credit-score percentage within each Credit_Mix category:")
display(credit_mix_share.round(1))

print("Median income, payment delay, and outstanding debt by credit-score class:")
summary_by_score = clean_df.groupby("Credit_Score")[[
    "Annual_Income", "Delay_from_due_date", "Outstanding_Debt"
]].median().reindex(CLASS_ORDER)
display(summary_by_score.round(2))

From these checks, I found that `Standard` is the largest target class and `Good` is the smallest. I also found that the target-class proportions differ across `Credit_Mix` categories, and the median payment delay differs across score classes. These patterns helped me decide which variables to inspect during modeling. The patterns are not proof of causation, and the strong relationship with `Credit_Mix` needs a feature-definition check before any real-world use.

## 10. Prepare the input features

I used `Credit_Score` as the target. I removed row/customer identifiers and personal identifiers from the model. I also removed `Month`, because each customer has repeated monthly records and I want to test generalization to customers not seen in training. I left out `Type_of_Loan` because it contains thousands of different text combinations; extracting the individual loan types would be a separate feature-engineering step.

I identified numerical and categorical columns from their data types so the same lists are used consistently by both models.

In [ ]:
target = "Credit_Score"
columns_to_remove = ["ID", "Customer_ID", "Name", "SSN", "Month", "Type_of_Loan"]

X = clean_df.drop(columns=[target] + columns_to_remove)
y = clean_df[target]
customer_groups = clean_df["Customer_ID"]

numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

print("Number of input features:", X.shape[1])
print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

## 11. Split the data by customer

The dataset has multiple monthly rows for each customer. I used `GroupShuffleSplit` so every row from a customer stays in either the training set or the test set, but not both. This makes the test set a better check of performance on unseen customers than a random row-level split.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_indices, test_indices = next(splitter.split(X, y, groups=customer_groups))

X_train = X.iloc[train_indices].copy()
X_test = X.iloc[test_indices].copy()
y_train = y.iloc[train_indices].copy()
y_test = y.iloc[test_indices].copy()

train_customers = set(customer_groups.iloc[train_indices])
test_customers = set(customer_groups.iloc[test_indices])
customer_overlap = train_customers.intersection(test_customers)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training customers:", len(train_customers))
print("Testing customers:", len(test_customers))
print("Customers in both sets:", len(customer_overlap))

## 12. Create a simple baseline

Before training a more complex model, I used a `DummyClassifier` that always predicts the most common class in the training data. This is my baseline: the other models should perform better than this simple approach, particularly on macro F1.

In [ ]:
baseline_model = DummyClassifier(strategy="most_frequent")
baseline_model.fit(X_train, y_train)
baseline_predictions = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_macro_f1 = f1_score(y_test, baseline_predictions, average="macro")

print("Baseline accuracy:", round(baseline_accuracy, 4))
print("Baseline macro F1:", round(baseline_macro_f1, 4))
print("Baseline classification report:")
print(classification_report(y_test, baseline_predictions, labels=CLASS_ORDER, zero_division=0))

## 13. Prepare the data for Random Forest

I handled missing values inside a preprocessing pipeline. For numerical columns, I used the median; for categorical columns, I used the most common category and then one-hot encoding. Keeping these steps in the pipeline means that the imputation values are learned from the training data rather than from the test data.

In [ ]:
numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features)
])

## 14. Model 1 – Random Forest

I used Random Forest as my main model because it can learn non-linear patterns and interactions between features. I used 100 trees and class weighting to reduce the model's tendency to focus only on the most frequent class. I kept the settings simple instead of running an extensive tuning search, so the workflow is easier to explain and reproduce.

In [ ]:
random_forest_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

random_forest_model.fit(X_train, y_train)
rf_predictions = random_forest_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_macro_f1 = f1_score(y_test, rf_predictions, average="macro")

print("Random Forest accuracy:", round(rf_accuracy, 4))
print("Random Forest macro F1:", round(rf_macro_f1, 4))
print("Random Forest classification report:")
print(classification_report(y_test, rf_predictions, labels=CLASS_ORDER, zero_division=0))

## 15. Model 2 – Logistic Regression

I used Logistic Regression as a second model for comparison. It is a simpler, linear classification method. I standardized the numerical features for this model because Logistic Regression is more sensitive to the scale of input values. The categorical variables are handled in the same way as for Random Forest.

In [ ]:
numeric_preprocessor_lr = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor_lr = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_lr = ColumnTransformer(transformers=[
    ("numeric", numeric_preprocessor_lr, numeric_features),
    ("categorical", categorical_preprocessor_lr, categorical_features)
])

logistic_model = Pipeline(steps=[
    ("preprocessing", preprocessor_lr),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)

logistic_accuracy = accuracy_score(y_test, logistic_predictions)
logistic_macro_f1 = f1_score(y_test, logistic_predictions, average="macro")

print("Logistic Regression accuracy:", round(logistic_accuracy, 4))
print("Logistic Regression macro F1:", round(logistic_macro_f1, 4))
print("Logistic Regression classification report:")
print(classification_report(y_test, logistic_predictions, labels=CLASS_ORDER, zero_division=0))

## 16. Compare the baseline and the two models

I compared accuracy and macro F1 on the same customer-held-out test set. Accuracy shows the overall share of correct predictions. Macro F1 calculates an F1 score for each class and gives the three classes equal weight, so I use it to compare the models alongside accuracy.

In [ ]:
metrics_df = pd.DataFrame([
    {"Model": "Majority-class baseline", "Accuracy": baseline_accuracy, "Macro F1": baseline_macro_f1},
    {"Model": "Random Forest", "Accuracy": rf_accuracy, "Macro F1": rf_macro_f1},
    {"Model": "Logistic Regression", "Accuracy": logistic_accuracy, "Macro F1": logistic_macro_f1}
])

display(metrics_df.round(4))

ax = metrics_df.set_index("Model")[["Accuracy", "Macro F1"]].plot(kind="bar", figsize=(9, 5))
ax.set_title("Model Performance Comparison")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 17. Check the effect of `Credit_Mix`

`Credit_Mix` showed a strong relationship with the target in the charts. I therefore ran a sensitivity check by training another Random Forest after removing this one feature. Both versions use the same training and test customers. A change in performance shows how much the model relies on this feature; it does **not** prove that the feature leaks the target. Its source and availability at prediction time still need to be confirmed.

In [ ]:
X_train_no_mix = X_train.drop(columns=["Credit_Mix"])
X_test_no_mix = X_test.drop(columns=["Credit_Mix"])

numeric_features_no_mix = X_train_no_mix.select_dtypes(include="number").columns.tolist()
categorical_features_no_mix = X_train_no_mix.select_dtypes(exclude="number").columns.tolist()

numeric_preprocessor_no_mix = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_preprocessor_no_mix = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_no_mix = ColumnTransformer(transformers=[
    ("numeric", numeric_preprocessor_no_mix, numeric_features_no_mix),
    ("categorical", categorical_preprocessor_no_mix, categorical_features_no_mix)
])

random_forest_no_mix = Pipeline(steps=[
    ("preprocessing", preprocessor_no_mix),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

random_forest_no_mix.fit(X_train_no_mix, y_train)
no_mix_predictions = random_forest_no_mix.predict(X_test_no_mix)

no_mix_accuracy = accuracy_score(y_test, no_mix_predictions)
no_mix_macro_f1 = f1_score(y_test, no_mix_predictions, average="macro")

credit_mix_check = pd.DataFrame([
    {"Model": "Random Forest with Credit_Mix", "Accuracy": rf_accuracy, "Macro F1": rf_macro_f1},
    {"Model": "Random Forest without Credit_Mix", "Accuracy": no_mix_accuracy, "Macro F1": no_mix_macro_f1}
])

display(credit_mix_check.round(4))

The comparison above is a sensitivity check, not a final judgement on the feature. If performance changes when I remove `Credit_Mix`, I know the model depends on it to some extent. Before the model is considered for any real process, I would confirm when this feature is created and whether it is available independently of the credit-score label.

## 18. Final model and confusion matrix

For this project, I select the model with the higher macro F1 among Random Forest and Logistic Regression. I then review the confusion matrix and class-level precision, recall, and F1 scores. The confusion matrix shows which classes the model confuses with one another.

The two models were compared on the same test set, so this is a straightforward project comparison rather than an independent production validation. A separate validation set would make model selection stronger in a later version.

In [ ]:
if rf_macro_f1 >= logistic_macro_f1:
    final_model = random_forest_model
    final_predictions = rf_predictions
    final_model_name = "Random Forest"
else:
    final_model = logistic_model
    final_predictions = logistic_predictions
    final_model_name = "Logistic Regression"

print("Selected model:", final_model_name)
print("Final test accuracy:", round(accuracy_score(y_test, final_predictions), 4))
print("Final test macro F1:", round(f1_score(y_test, final_predictions, average="macro"), 4))

final_report = classification_report(
    y_test,
    final_predictions,
    labels=CLASS_ORDER,
    output_dict=True,
    zero_division=0
)
final_report_df = pd.DataFrame(final_report).T
print("Final classification report:")
display(final_report_df.round(4))

cm = confusion_matrix(y_test, final_predictions, labels=CLASS_ORDER)
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_ORDER)
cm_display.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix – " + final_model_name)
plt.tight_layout()
plt.show()

## 19. Check feature importance

I used permutation importance to check which original input columns the selected model relies on most for macro F1. The method shuffles one feature at a time and measures how the score changes. This describes the model's behaviour; it does not prove that a feature causes the credit-score outcome.

In [ ]:
importance_rows = X_test.sample(n=min(5000, len(X_test)), random_state=RANDOM_STATE)
importance_labels = y_test.loc[importance_rows.index]

importance_result = permutation_importance(
    final_model,
    importance_rows,
    importance_labels,
    scoring="f1_macro",
    n_repeats=3,
    random_state=RANDOM_STATE,
    n_jobs=1
)

feature_importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": importance_result.importances_mean
}).sort_values("Importance", ascending=False).reset_index(drop=True)

print("Top 10 features by permutation importance:")
display(feature_importance_df.head(10).round(4))

importance_to_plot = feature_importance_df.head(10).sort_values("Importance")
plt.figure(figsize=(8, 5))
plt.barh(importance_to_plot["Feature"], importance_to_plot["Importance"])
plt.title("Top Features by Permutation Importance")
plt.xlabel("Decrease in macro F1 after shuffling")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 20. My conclusion

I used the code below to report the metrics calculated from the current run, rather than manually inserting numbers into the conclusion. This makes the conclusion stay consistent if the input data or model results change.

In [ ]:
print("PROJECT CONCLUSION")
print("-------------------")
print(f"I analyzed {len(clean_df):,} records from {clean_df['Customer_ID'].nunique():,} customers.")
print(f"I used {X.shape[1]} input features: {len(numeric_features)} numerical and {len(categorical_features)} categorical.")
print(f"The customer-level split left {len(customer_overlap)} customers in both training and test sets.")
print(f"The majority-class baseline reached accuracy {baseline_accuracy:.3f} and macro F1 {baseline_macro_f1:.3f}.")
print(f"Random Forest reached accuracy {rf_accuracy:.3f} and macro F1 {rf_macro_f1:.3f}.")
print(f"Logistic Regression reached accuracy {logistic_accuracy:.3f} and macro F1 {logistic_macro_f1:.3f}.")
print(f"I selected {final_model_name} because it had the higher macro F1 in this comparison.")
print(f"The selected model's test accuracy was {accuracy_score(y_test, final_predictions):.3f} and its macro F1 was {f1_score(y_test, final_predictions, average='macro'):.3f}.")
print(f"Removing Credit_Mix changed Random Forest macro F1 to {no_mix_macro_f1:.3f}, compared with {rf_macro_f1:.3f} when it was included.")
print("These scores describe this dataset and split only. They do not establish real-world lending performance or fairness.")

## 21. Limitations and next steps

Based on the work completed, I would address the following points before using this model beyond an educational project:

1. **Feature definition:** Confirm the source and creation time of `Credit_Mix`, especially because the model's results change when it is removed.
2. **Data quality:** Confirm the meaning of placeholder values and the age values below 18 with the data provider. The age rule in this project is an assumption.
3. **Validation:** Test on a later time period or a separate customer group and use a separate validation set for model selection.
4. **Fairness and governance:** Review demographic and proxy features, calibration, performance across customer groups, and applicable rules before any real financial decision.
5. **Model scope:** The target is `Credit_Score`; this project does not detect banking fraud.

I would not use this model on its own to approve or reject an actual credit application. The supplied dataset and this test do not provide enough evidence for that decision.

## 22. Save the model and reports

I saved the complete model pipeline, including preprocessing, so that it can accept the same raw input columns when loaded again. I also saved the model comparison, final classification report, and feature-importance table in my project folder.

In [ ]:
output_dir = Path("/content/drive/MyDrive/Credit_Score_Project")
output_dir.mkdir(parents=True, exist_ok=True)

model_path = output_dir / "credit_score_prediction_model.joblib"
metrics_path = output_dir / "credit_score_model_comparison.csv"
report_path = output_dir / "credit_score_classification_report.csv"
importance_path = output_dir / "credit_score_feature_importance.csv"

joblib.dump(final_model, model_path)
metrics_df.to_csv(metrics_path, index=False)
final_report_df.to_csv(report_path)
feature_importance_df.to_csv(importance_path, index=False)

print("Saved model:", model_path)
print("Saved model comparison:", metrics_path)
print("Saved classification report:", report_path)
print("Saved feature importance:", importance_path)

## 23. Reload the model and test a few rows

I reloaded the saved model and predicted a few rows from the held-out test set. This is a technical check that the saved pipeline can be loaded again and make predictions. These example predictions are not intended for real credit decisions.

In [ ]:
loaded_model = joblib.load(model_path)
example_rows = X_test.head(5).copy()
example_predictions = loaded_model.predict(example_rows)

sanity_check = pd.DataFrame({
    "Actual_Credit_Score": y_test.loc[example_rows.index].values,
    "Predicted_Credit_Score": example_predictions
})

display(sanity_check)
print("Reload-and-predict check completed successfully.")

## Final submission checklist

Before I submit the project, I will confirm that:

- [ ] I ran all cells from the beginning in Google Colab.
- [ ] The dataset loaded from `Credit_Score_Project/dataset-2.csv`.
- [ ] The output confirms that no customers appear in both the training and test sets.
- [ ] The charts, model comparison, classification report, confusion matrix, and `Credit_Mix` check are present.
- [ ] I reviewed the conclusion and limitations and can explain them in my own words.
- [ ] The notebook is saved with its outputs in Google Drive.
- [ ] The trained model and CSV reports are present in the project folder.
- [ ] My presentation explains the problem, data cleaning, charts, customer-level split, models, metrics, findings, and limitations.

**End of project notebook.**